In [1]:
import os
import json
from pathlib import Path
import pandas as pd

OUTPUT_ROOT = Path('/Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/processed_10hz/v2')
MODEL_ROOT  = Path('/Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/model_outputs/v2')
AE_DIR      = MODEL_ROOT / '05_ae_comparison'
FOLDS       = [f'fold_{i:02d}' for i in range(1, 6)]

print(f'OUTPUT_ROOT exists: {OUTPUT_ROOT.exists()}')
print(f'AE_DIR exists     : {AE_DIR.exists()}')

OUTPUT_ROOT exists: True
AE_DIR exists     : True


In [2]:
# 1. PREPROCESSED PARQUETS PER FOLD
EXPECTED_SPLITS = ['train', 'val', 'test']
EXPECTED_SCALES = ['scaled', 'unscaled']

rows = []
for fold in FOLDS:
    fold_dir = OUTPUT_ROOT / fold
    row = {'fold': fold, 'dir_exists': fold_dir.exists()}
    if fold_dir.exists():
        files = list(fold_dir.glob('*.parquet'))
        row['parquet_files'] = sorted([f.name for f in files])
        row['n_files'] = len(files)
        for split in EXPECTED_SPLITS:
            for scale in EXPECTED_SCALES:
                found = any(split in f.name and scale in f.name for f in files)
                row[f'{split}_{scale}'] = '✓' if found else '✗'
    else:
        row['parquet_files'] = []
        row['n_files'] = 0
        for split in EXPECTED_SPLITS:
            for scale in EXPECTED_SCALES:
                row[f'{split}_{scale}'] = '✗'
    rows.append(row)

df_fold = pd.DataFrame(rows)
cols_show = ['fold', 'dir_exists', 'n_files',
             'train_scaled', 'train_unscaled',
             'val_scaled',   'val_unscaled',
             'test_scaled',  'test_unscaled']
print(df_fold[cols_show].to_string(index=False))

print('\n=== Raw file listing per fold ===')
for _, r in df_fold.iterrows():
    print(f"\n{r['fold']}:")
    for fname in r['parquet_files']:
        print(f'  {fname}')

   fold  dir_exists  n_files train_scaled train_unscaled val_scaled val_unscaled test_scaled test_unscaled
fold_01        True        6            ✓              ✗          ✓            ✗           ✓             ✗
fold_02        True        6            ✓              ✗          ✓            ✗           ✓             ✗
fold_03        True        6            ✓              ✗          ✓            ✗           ✓             ✗
fold_04        True        6            ✓              ✗          ✓            ✗           ✓             ✗
fold_05        True        6            ✓              ✗          ✓            ✗           ✓             ✗

=== Raw file listing per fold ===

fold_01:
  test_no_dr_interpolated_v5_voltage_kfold5_fold_01.parquet
  test_no_dr_scaled_v5_voltage_kfold5_fold_01.parquet
  train_no_dr_interpolated_v5_voltage_kfold5_fold_01.parquet
  train_no_dr_scaled_v5_voltage_kfold5_fold_01.parquet
  val_no_dr_interpolated_v5_voltage_kfold5_fold_01.parquet
  val_no_dr_scaled_v5_vo

In [3]:
# 2. TRANSFORMER-AE FLIGHT SCORE PARQUETS
print('=== Transformer-AE flight score files ===')
score_files = sorted(AE_DIR.glob('flight_scores_transformer_ae_fold_*.parquet'))
if score_files:
    for f in score_files:
        df_s = pd.read_parquet(f)
        print(f'{f.name}')
        print(f'  shape: {df_s.shape}  |  columns: {list(df_s.columns)}')
        print(f'  splits present: {df_s["split"].unique() if "split" in df_s.columns else "no split column"}')
        print()
else:
    print(f'No score files found in {AE_DIR}')

=== Transformer-AE flight score files ===
flight_scores_transformer_ae_fold_01.parquet
  shape: (34, 7)  |  columns: ['log_id', 'dataset_label', 'mean_score', 'binary_label', 'fold', 'model', 'threshold']
  splits present: no split column

flight_scores_transformer_ae_fold_02.parquet
  shape: (34, 7)  |  columns: ['log_id', 'dataset_label', 'mean_score', 'binary_label', 'fold', 'model', 'threshold']
  splits present: no split column

flight_scores_transformer_ae_fold_03.parquet
  shape: (33, 7)  |  columns: ['log_id', 'dataset_label', 'mean_score', 'binary_label', 'fold', 'model', 'threshold']
  splits present: no split column

flight_scores_transformer_ae_fold_04.parquet
  shape: (33, 7)  |  columns: ['log_id', 'dataset_label', 'mean_score', 'binary_label', 'fold', 'model', 'threshold']
  splits present: no split column

flight_scores_transformer_ae_fold_05.parquet
  shape: (33, 7)  |  columns: ['log_id', 'dataset_label', 'mean_score', 'binary_label', 'fold', 'model', 'threshold']
  s

In [4]:
# 3. MODEL WEIGHTS
print('=== Saved Transformer-AE model weights ===')
weight_files = sorted(AE_DIR.glob('*.pt')) + sorted(AE_DIR.glob('**/*.pt'))
if weight_files:
    for f in weight_files:
        size_mb = f.stat().st_size / 1e6
        print(f'  {f.relative_to(AE_DIR)}  ({size_mb:.2f} MB)')
else:
    print(f'  No .pt weight files found under {AE_DIR}')

=== Saved Transformer-AE model weights ===
  cnn_lstm_ae_fold_01.pt  (0.34 MB)
  cnn_lstm_ae_fold_02.pt  (0.34 MB)
  cnn_lstm_ae_fold_03.pt  (0.34 MB)
  cnn_lstm_ae_fold_04.pt  (0.34 MB)
  cnn_lstm_ae_fold_05.pt  (0.34 MB)
  lstm_ae_fold_01.pt  (0.53 MB)
  lstm_ae_fold_02.pt  (0.53 MB)
  lstm_ae_fold_03.pt  (0.53 MB)
  lstm_ae_fold_04.pt  (0.53 MB)
  lstm_ae_fold_05.pt  (0.53 MB)
  transformer_ae_fold_01.pt  (0.47 MB)
  transformer_ae_fold_02.pt  (0.47 MB)
  transformer_ae_fold_03.pt  (0.47 MB)
  transformer_ae_fold_04.pt  (0.47 MB)
  transformer_ae_fold_05.pt  (0.47 MB)
  cnn_lstm_ae_fold_01.pt  (0.34 MB)
  cnn_lstm_ae_fold_02.pt  (0.34 MB)
  cnn_lstm_ae_fold_03.pt  (0.34 MB)
  cnn_lstm_ae_fold_04.pt  (0.34 MB)
  cnn_lstm_ae_fold_05.pt  (0.34 MB)
  lstm_ae_fold_01.pt  (0.53 MB)
  lstm_ae_fold_02.pt  (0.53 MB)
  lstm_ae_fold_03.pt  (0.53 MB)
  lstm_ae_fold_04.pt  (0.53 MB)
  lstm_ae_fold_05.pt  (0.53 MB)
  transformer_ae_fold_01.pt  (0.47 MB)
  transformer_ae_fold_02.pt  (0.47 MB)
  tr

In [5]:
# 4. SCALER + FEATURE_COLS
print('=== Scaler + feature_cols availability ===')
for fold in FOLDS:
    fold_dir = OUTPUT_ROOT / fold
    scaler = list(fold_dir.glob('*.joblib')) if fold_dir.exists() else []
    fcols  = list(fold_dir.glob('feature_cols*.json')) if fold_dir.exists() else []
    print(f"{fold}  scaler: {'✓ ' + scaler[0].name if scaler else '✗'}  |  "
          f"feature_cols: {'✓ ' + fcols[0].name if fcols else '✗'}")

=== Scaler + feature_cols availability ===
fold_01  scaler: ✓ scaler_fold_01.joblib  |  feature_cols: ✓ feature_cols_fold_01.json
fold_02  scaler: ✓ scaler_fold_02.joblib  |  feature_cols: ✓ feature_cols_fold_02.json
fold_03  scaler: ✓ scaler_fold_03.joblib  |  feature_cols: ✓ feature_cols_fold_03.json
fold_04  scaler: ✓ scaler_fold_04.joblib  |  feature_cols: ✓ feature_cols_fold_04.json
fold_05  scaler: ✓ scaler_fold_05.joblib  |  feature_cols: ✓ feature_cols_fold_05.json


In [6]:
# 5. VERDICT
has_val   = df_fold['val_scaled'].eq('✓').all()
has_wts   = bool(weight_files)

print('=' * 60)
if has_val and has_wts:
    print('OOF scoring FULLY FEASIBLE')
    print('Load each fold .pt, score val split, concat val+test → full coverage')
elif has_val and not has_wts:
    print('Val parquets exist but NO model weights saved')
    print('Must re-run notebook 05 with checkpoint saving enabled')
elif not has_val and has_wts:
    print('Weights exist but val parquets MISSING')
    print('Regenerate val splits from preprocessing notebook')
else:
    print('Neither val parquets nor weights found — full re-run needed')

if not has_val:
    missing = df_fold.loc[df_fold['val_scaled'] != '✓', 'fold'].tolist()
    print(f'Folds missing val parquets: {missing}')

OOF scoring FULLY FEASIBLE
Load each fold .pt, score val split, concat val+test → full coverage


In [8]:
import pandas as pd
from pathlib import Path

ENRICHED_DIR = Path('/Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/model_outputs/v2/07_enriched_parquets')

fold = 'fold_01'
df   = pd.read_parquet(ENRICHED_DIR / f'test_enriched_{fold}.parquet')

print(f'Shape: {df.shape}')
print(f'\nAll columns ({len(df.columns)}):')
for c in df.columns:
    print(f'  {c}')

print(f'\nPower-related:')
power_cols = [c for c in df.columns if any(k in c.lower() for k in ['power','watt','wh','energy'])]
print(power_cols if power_cols else '  NONE')

print(f'\nAnomaly-related:')
anom_cols = [c for c in df.columns if any(k in c.lower() for k in ['anomaly','score','flag'])]
print(anom_cols if anom_cols else '  NONE')

print(f'\nNull counts (anomaly cols):')
print(df[anom_cols].isna().sum().to_string() if anom_cols else '  NONE')

print(f'\nSample stats (power + anomaly cols):')
show = power_cols + anom_cols
if show:
    print(df[show].describe().round(4).to_string())

print(f'\ndataset_label present: {"dataset_label" in df.columns}')
if 'dataset_label' in df.columns:
    print(f'  values: {df["dataset_label"].value_counts().to_dict()}')

FileNotFoundError: [Errno 2] No such file or directory: '/Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/model_outputs/v2/07_enriched_parquets/test_enriched_fold_01.parquet'

In [11]:
from pathlib import Path
import pandas as pd

fold_dir = Path('/Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/processed_10hz/v2/fold_01')
df = pd.read_parquet(fold_dir / 'train_no_dr_scaled_v5_voltage_kfold5_fold_01.parquet')

print(df['dataset_label'].dtype)
print(df['dataset_label'].value_counts())
print(df.columns.tolist())

object
dataset_label
normal      466861
abnormal    198355
Name: count, dtype: int64
['log_id', 'dataset_label', 'battery_status__cell_count', 'battery_status__current_a', 'battery_status__current_filtered_a', 'battery_status__discharged_mah', 'battery_status__remaining', 'battery_status__scale', 'battery_status__voltage_filtered_v', 'battery_status__voltage_v', 'cumulative_energy_wh', 'power_w', 'time_s', 'is_idle', 'power_w_rolling_std_3s', 'voltage_per_cell', 'voltage_filtered_per_cell', 'voltage_per_cell_suspicious', 'target_energy_next_60s_wh']


In [12]:
fold_dir = Path('/Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/processed_10hz/v2/fold_01')
df = pd.read_parquet(fold_dir / 'train_no_dr_scaled_v5_voltage_kfold5_fold_01.parquet')

print('voltage_per_cell_suspicious values:')
print(df['voltage_per_cell_suspicious'].value_counts())

print('\ncross-tab label vs suspicious:')
print(pd.crosstab(df['dataset_label'], df['voltage_per_cell_suspicious']))

print('\nnormal + suspicious==0 rows:')
clean = df[(df['dataset_label'] == 'normal') & (df['voltage_per_cell_suspicious'] == 0)]
print(f'  {len(clean)} / {len(df[df["dataset_label"]=="normal"])} normal rows kept')
print(f'  {clean["log_id"].nunique()} flights')

voltage_per_cell_suspicious values:
voltage_per_cell_suspicious
-0.425188    586639
 2.351902     78577
Name: count, dtype: int64

cross-tab label vs suspicious:
voltage_per_cell_suspicious  -0.425188   2.351902
dataset_label                                    
abnormal                        191257       7098
normal                          395382      71479

normal + suspicious==0 rows:
  0 / 466861 normal rows kept
  0 flights


In [13]:
# check unscaled to confirm
fold_dir = Path('/Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/processed_10hz/v2/fold_01')
df_raw = pd.read_parquet(fold_dir / 'train_no_dr_interpolated_v5_voltage_kfold5_fold_01.parquet')

print('unscaled voltage_per_cell_suspicious:')
print(df_raw['voltage_per_cell_suspicious'].value_counts())

unscaled voltage_per_cell_suspicious:
voltage_per_cell_suspicious
0    586639
1     78577
Name: count, dtype: int64
